# Agency: mi revisión SQL antes de Silver

Este es mi punto de entrada para revisar agency con calma. Voy a responder una pregunta por bloque; puedo dejar las consultas largas para después de entender las pequeñas.

Sigo explorando Bronze para decidir el contrato Silver. Este archivo no crea tablas ni limpia registros; solo prepara widgets de sesión y consultas de lectura. Todavía no he ejecutado esta guía nueva en Databricks. Los resultados de 2025 que ya obtuve con Genie están en el informe del proyecto.

Mi recorrido es: ver columnas, mirar filas, comprobar identidad, revisar ausencias e IDs repetidos y consultar el encabezado de origen. Después decidiré qué reglas necesito.

Las consultas cortas son ejercicios preparados. En el apéndice guardé las dos consultas largas que sí ejecuté con Genie para 2025; no necesito repetirlas solo para estudiar el proceso.


## Bloque 0 — ¿Qué snapshot voy a mirar?

Uso widgets como casillas de configuración. Los valores iniciales corresponden a mi entorno y al snapshot de 2025; cambiarlos no modifica la tabla almacenada.

`:feed_version` y `:snapshot_sha256` toman los valores de esas casillas. `identifier(:agency_table)` interpreta el nombre de tabla configurado. Antes de ejecutar la guía comprobaré que el entorno admita esta sintaxis de widgets, disponible desde Databricks Runtime 15.2.

Empiezo por 2025. Si más adelante cambio de snapshot, cambiaré versión y hash juntos y volveré a ejecutar desde aquí. Los apéndices siguen fijados a 2025, independientemente de los widgets.


In [ ]:
# BLOQUE 0 — Configuración de sesión; no escribe archivos ni tablas.
dbutils.widgets.text("agency_table", "workspace.gtfs_bogota.bronze_gtfs_agency", "Tabla Bronze agency")
dbutils.widgets.text("schema_audit_table", "workspace.gtfs_bogota.audit_gtfs_source_schema_observations", "Auditoría de encabezados")
dbutils.widgets.text("feed_version", "2025-10-28", "Versión GTFS")
dbutils.widgets.text(
    "snapshot_sha256",
    "07ba2ba9bb3b5bc1e38357355956757ce84fc68e6d05b74577bf017e189ab952",
    "SHA-256 del snapshot",
)
print("Versión seleccionada:", dbutils.widgets.get("feed_version"))
print("Hash seleccionado:", dbutils.widgets.get("snapshot_sha256"))


## Bloque 1 — ¿Qué columnas tengo y cómo están almacenadas?

Con `describe` voy a mirar los nombres y tipos actuales. Espero campos fuente como string porque así los conservo en Bronze. Ese tipo de almacenamiento no decide automáticamente el tipo que daré a otra entidad en Silver.

El esquema pertenece a la tabla histórica completa. Que aquí aparezca agency_fare_url no demuestra que la columna existiera en el archivo de 2025; lo revisaré en el bloque 6.


In [ ]:
%sql
-- BLOQUE 1 — Esquema actual; solo lee metadatos.
describe identifier(:agency_table);


## Bloque 2 — ¿Qué representa una fila de agency?

Voy a mirar hasta 20 registros del snapshot. En la consulta previa de 2025 recuperé las siete filas completas. Primero compararé agency_id y agency_name para entender la identidad de cada registro.

Los campos source_* me sirven para rastrear el dato. Compartir teléfono o URL no convierte dos IDs diferentes en duplicados, ni me permite interpretar automáticamente las filas como siete empresas operadoras distintas.


In [ ]:
%sql
-- BLOQUE 2 — Muestra acotada del snapshot, sin cambiar valores.
select
    agency_id,
    agency_name,
    agency_url,
    agency_timezone,
    agency_lang,
    agency_phone,
    agency_fare_url
from identifier(:agency_table)
where source_feed_version = :feed_version
  and source_sha256 = :snapshot_sha256
order by agency_id
limit 20;


## Bloque 3 — ¿Cuántas filas e IDs distintos tengo?

`count(*)` cuenta filas. `count(distinct agency_id)` cuenta identificadores diferentes y omite null. Comparo ambas cantidades como primera pista; todavía necesito revisar IDs vacíos, conflictos y significado antes de aprobar una clave.

En 2025 obtuve siete en ambos conteos. Lo registro como resultado de esa versión y hash, no como una cantidad que deba exigir a todos los feeds futuros.


In [ ]:
%sql
-- BLOQUE 3 — Primera comprobación de identidad.
select
    count(*) as filas,
    count(distinct agency_id) as ids_distintos_no_nulos
from identifier(:agency_table)
where source_feed_version = :feed_version
  and source_sha256 = :snapshot_sha256;


## Bloque 4 — ¿Hay un ID nulo, vacío o compuesto solo por espacios?

Empiezo con una columna: agency_id. Distingo null, cadena vacía y texto que solo contiene espacios. El patrón de expresión regular detecta espacios, tabulaciones u otros caracteres de espacio sin contenido útil.

Si el lector CSV ya convirtió un vacío a null, Bronze no me permite reconstruir esa diferencia original por sí sola. Después aplicaré estas preguntas a los demás campos. La consulta del apéndice es más larga porque reúne todos esos conteos.


In [ ]:
%sql
-- BLOQUE 4 — Tres categorías exclusivas para agency_id.
select
    count(*) as filas,
    count_if(agency_id is null) as ids_nulos,
    count_if(agency_id = '') as ids_vacios,
    count_if(agency_id rlike r'^\s+$') as ids_solo_espacios
from identifier(:agency_table)
where source_feed_version = :feed_version
  and source_sha256 = :snapshot_sha256;


## Bloque 5 — ¿Se repite algún identificador?

Agrupo por agency_id y muestro solo los grupos con más de una fila. Si la salida está vacía, no encontré IDs repetidos para ese snapshot.

Si aparece un grupo, revisaré si son copias iguales o si el mismo ID tiene atributos distintos. Dejé esa comprobación en el apéndice y en el perfilador PySpark. Aquí solo detecto repeticiones: no elimino registros ni apruebo la calidad de un ID únicamente por este conteo.


In [ ]:
%sql
-- BLOQUE 5 — Detección de claves repetidas, sin deduplicar.
select
    agency_id,
    count(*) as filas_por_id
from identifier(:agency_table)
where source_feed_version = :feed_version
  and source_sha256 = :snapshot_sha256
group by agency_id
having count(*) > 1
order by agency_id;


## Bloque 6 — ¿La columna estaba en el archivo publicado?

Consulto la observación de encabezados que guardó Bronze. Para 2025 encontré seis columnas y agency_fare_url no estaba entre ellas; por eso sus null en la tabla histórica requieren una interpretación distinta.

Esto es como comparar dos formularios: si uno ni siquiera incluye la casilla de teléfono, no puedo decir que alguien la dejó vacía. En los datos también separo una columna ausente de una celda sin valor. No copiaré información de 2026 para rellenar esa ausencia.


In [ ]:
%sql
-- BLOQUE 6 — Evidencia de presencia en el encabezado del snapshot.
select
    source_feed_version,
    source_file_name,
    source_columns_json
from identifier(:schema_audit_table)
where source_feed_version = :feed_version
  and source_sha256 = :snapshot_sha256
  and source_file_name = 'agency.txt'
limit 2;


## Mi pausa antes de transformar

1. Necesito explicar qué representa una fila según el modelo del proveedor y situar su identidad dentro del snapshot.
2. Mantendría los IDs y teléfonos conocidos como string: los uso para identificar o contactar, no para hacer aritmética.
3. Los controles de 2025 no mostraron una necesidad de completar ni deduplicar los registros.
4. Me falta revisar agency de 2026 y definir formatos y reglas antes de crear la transformación Silver.

No quiero acumular tablas o controles sin entender el anterior. El perfilador 08_profile_gtfs_reference_entities.ipynb me servirá para automatizar después estas preguntas con PySpark; no necesito ejecutar ambas opciones como dos requisitos distintos.


## Apéndice A — La consulta completa de campos que ejecuté con Genie

Puedo dejar esta lectura para después. Reuní las mismas preguntas para siete columnas en una sola consulta: `snapshot` selecciona la versión, `cells` pone los campos en una estructura común y la última parte calcula las métricas.

Conservo el SQL histórico exacto, incluidos los comentarios del archivo original. Está fijado a 2025 y al entorno de esa revisión; no usa los widgets anteriores. Lo guardo como evidencia de diagnóstico, no como transformación Silver.


In [ ]:
%sql
-- Perfilado de solo lectura ejecutado para agency, snapshot 2025-10-28.
-- Entorno actual: workspace.gtfs_bogota; no es un pipeline genérico.
-- No usar estos campos de duplicados para otra versión sin revisar su encabezado.
-- El hash y la fecha fijan el snapshot; no se fijó una versión Delta.
-- Consulta 1: métricas de texto. agency_fare_url no estuvo publicada en 2025.
with snapshot as (
select agency_id, agency_name, agency_url, agency_timezone, agency_lang, agency_phone, agency_fare_url from workspace.gtfs_bogota.bronze_gtfs_agency where source_feed_version = '2025-10-28' and source_sha256 = '07ba2ba9bb3b5bc1e38357355956757ce84fc68e6d05b74577bf017e189ab952'
), cells as (
select stack(7, 'agency_id', agency_id, 'agency_name', agency_name, 'agency_url', agency_url, 'agency_timezone', agency_timezone, 'agency_lang', agency_lang, 'agency_phone', agency_phone, 'agency_fare_url', agency_fare_url) as (field_name, field_value)
from snapshot
)
select field_name, count(*) as snapshot_rows,
sum(case when field_value is null then 1 else 0 end) as null_count,
sum(case when field_value = '' then 1 else 0 end) as empty_count,
sum(case when field_value rlike r'^\s+$' then 1 else 0 end) as whitespace_only_count,
sum(case when field_value is not null and not(field_value rlike r'^\s*$') then 1 else 0 end) as non_blank_count,
count(distinct case when field_value is not null and not(field_value rlike r'^\s*$') then field_value end) as distinct_non_blank_count,
sum(case when field_value is not null and not(field_value rlike r'^\s*$') and field_value <> regexp_replace(field_value, r'^\s+|\s+$', '') then 1 else 0 end) as edge_whitespace_count
from cells
group by field_name
order by field_name;


## Apéndice B — La consulta completa de duplicados que ejecuté con Genie

También es una lectura opcional. Cada bloque intermedio responde una pregunta: IDs repetidos, variantes de una fila, posibles colisiones al quitar espacios y resumen final. No hago limpieza; comparo las seis columnas publicadas de 2025 y excluyo la procedencia técnica.

Conservo estas dos consultas en databricks/sql/silver/agency_profile_2025.sql como evidencia original. Para estudiar el recorrido puedo quedarme en este notebook, sin saltar entre archivos.


In [ ]:
%sql
-- Consulta 2: duplicados de negocio con las seis columnas publicadas.
with snapshot as (
select agency_id, agency_name, agency_url, agency_timezone, agency_lang, agency_phone, agency_fare_url from workspace.gtfs_bogota.bronze_gtfs_agency where source_feed_version = '2025-10-28' and source_sha256 = '07ba2ba9bb3b5bc1e38357355956757ce84fc68e6d05b74577bf017e189ab952'
), key_counts as (
select agency_id, count(*) as row_count,
count(distinct struct(agency_id, agency_name, agency_url, agency_timezone, agency_lang, agency_phone)) as business_variants
from snapshot
where agency_id is not null and not(agency_id rlike r'^\s*$')
group by agency_id
), trim_keys as (
select regexp_replace(agency_id, r'^\s+|\s+$', '') as id_after_trim,
count(distinct agency_id) as original_variants
from snapshot
where agency_id is not null and not(agency_id rlike r'^\s*$')
group by regexp_replace(agency_id, r'^\s+|\s+$', '')
), overall as (
select count(*) as total_rows,
count(*) - count(distinct struct(agency_id, agency_name, agency_url, agency_timezone, agency_lang, agency_phone)) as identical_extra_rows,
sum(case when agency_id is null or agency_id rlike r'^\s*$' then 1 else 0 end) as blank_id_rows
from snapshot
), key_summary as (
select count(*) as distinct_non_blank_ids,
count(case when row_count > 1 then 1 end) as repeated_id_groups,
count(case when business_variants > 1 then 1 end) as conflicting_id_groups
from key_counts
), trim_summary as (
select count(case when original_variants > 1 then 1 end) as potential_trim_collision_groups
from trim_keys
)
select overall.total_rows, overall.identical_extra_rows, overall.blank_id_rows,
key_summary.distinct_non_blank_ids, key_summary.repeated_id_groups, key_summary.conflicting_id_groups,
trim_summary.potential_trim_collision_groups
from overall cross join key_summary cross join trim_summary;


## Referencias y estado de mi revisión

- [Widgets](https://docs.databricks.com/aws/en/notebooks/widgets).
- [IDENTIFIER](https://docs.databricks.com/aws/en/sql/language-manual/sql-ref-names-identifier-clause).
- Informe agregado en docs/agency_profile_2025.md, dentro del repositorio.

Los apéndices conservan las consultas que ya comprobé. Todavía no he ejecutado en Databricks la guía nueva con widgets y consultas cortas. No guardé resultados de ejecución dentro del notebook ni creé tablas Silver.
